# Week 17 — Phase 4: Temporal Analysis (Part 1 of Hazard Mapping)
Extends Week 16's validated temporal check (built on the clean, restored
threshold=1.4 QuakeMigrate data) to all three catalogues: daily/weekly
seismicity rate plots and a full cross-catalogue comparison of the 4
September 2019 mainshock window.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

PROJECT_DIR = os.path.expanduser("~/tvz_project")
CATALOG_DIR = os.path.join(PROJECT_DIR, "catalogs")
OUT_DIR = os.path.join(CATALOG_DIR, "phase4")
FIGURES_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

WINDOW_START = pd.Timestamp("2019-08-01")
WINDOW_END = pd.Timestamp("2019-09-30")

In [ ]:
# Load all three catalogues with consistent origin_dt columns
eqt = pd.read_csv(os.path.join(CATALOG_DIR, "eqtransformer_nlloc_quality_filtered.csv"))
eqt["origin_dt"] = pd.to_datetime(eqt["origin_time"])

pn = pd.read_csv(os.path.join(CATALOG_DIR, "phasenet_nlloc_quality_filtered.csv"))
pn["origin_dt"] = pd.to_datetime(pn["origin_time"])

qm = pd.read_csv(os.path.join(CATALOG_DIR, "quakemigrate_quality_filtered.csv"))
qm["origin_dt"] = pd.to_datetime(qm["origin_time"], utc=True).dt.tz_localize(None)

catalogues = {"EQTransformer": eqt, "PhaseNet": pn, "QuakeMigrate": qm}
for name, df in catalogues.items():
    print(f"{name}: {len(df)} events, {df['origin_dt'].min()} to {df['origin_dt'].max()}")

In [ ]:
# Daily seismicity rate plot, all three catalogues, with the 4 September
# mainshock marked
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
colours = {"EQTransformer": "#1f77b4", "PhaseNet": "#ff7f0e", "QuakeMigrate": "#2ca02c"}
mainshock_time = pd.Timestamp("2019-09-04 07:02:51")

for ax, (name, df) in zip(axes, catalogues.items()):
    daily_counts = df.set_index("origin_dt").resample("D").size()
    full_range = pd.date_range(WINDOW_START, WINDOW_END, freq="D")
    daily_counts = daily_counts.reindex(full_range, fill_value=0)
    ax.bar(daily_counts.index, daily_counts.values, width=0.8, color=colours[name], alpha=0.8)
    ax.axvline(mainshock_time, color="red", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_ylabel(f"{name}\nevents/day")
    ax.grid(True, alpha=0.3)

axes[0].set_title("Daily event counts, all catalogues (red line: M5.2 mainshock, 4 Sep 2019)")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[-1].set_xlabel("Date")
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "temporal_rate_all_catalogues.png")
plt.savefig(fig_path, dpi=150)
plt.close()
print(f"Saved: {fig_path}")

In [ ]:
# Cross-catalogue comparison of the mainshock window, mirroring Week 16's
# validated approach for QuakeMigrate exactly (August rate vs 4-6 Sept
# window rate vs September-excluding-mainshock rate)
mainshock_window_start = pd.Timestamp("2019-09-04")
mainshock_window_end = pd.Timestamp("2019-09-07")  # exclusive, i.e. 4-6 Sept inclusive

rows = []
for name, df in catalogues.items():
    august = df[(df["origin_dt"] >= WINDOW_START) & (df["origin_dt"] < pd.Timestamp("2019-09-01"))]
    mainshock_window = df[(df["origin_dt"] >= mainshock_window_start) & (df["origin_dt"] < mainshock_window_end)]
    sept_total = df[(df["origin_dt"] >= pd.Timestamp("2019-09-01")) & (df["origin_dt"] <= WINDOW_END)]
    sept_excl = len(sept_total) - len(mainshock_window)

    rows.append({
        "catalogue": name,
        "august_events": len(august),
        "august_rate_per_day": len(august) / 31,
        "mainshock_window_events": len(mainshock_window),
        "mainshock_window_rate_per_day": len(mainshock_window) / 3,
        "sept_excl_mainshock_events": sept_excl,
        "sept_excl_mainshock_rate_per_day": sept_excl / 27,
        "ratio_sept_excl_to_august": (sept_excl / 27) / (len(august) / 31) if len(august) > 0 else np.nan,
    })

mainshock_comparison = pd.DataFrame(rows)
print(mainshock_comparison.to_string(index=False))

out_path = os.path.join(OUT_DIR, "week17_mainshock_window_comparison.csv")
mainshock_comparison.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")